# 节点11：InstructGPT 与 RLHF（2022）——让 AI 学会"听话"

> **论文**：Ouyang et al. 2022 "Training language models to follow instructions with human feedback"
> arXiv:2203.02155 | NeurIPS 2022

本 notebook 将手撕 RLHF 的三个核心组件：
1. **Bradley-Terry 偏好模型**：如何把「A 比 B 好」转化为可训练的目标
2. **Reward Model**：用偏好数据训练一个人类品味打分器
3. **PPO 中的 KL 惩罚**：为什么强化学习需要「不要走太远」的约束
4. **RLHF 全流程串联**：SFT → RM → PPO 的完整数据流

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
np.random.seed(42)
print('NumPy version:', np.__version__)

In [ ]:
# =============================================================
# Part 1: Bradley-Terry 偏好模型
# =============================================================

def sigmoid(x):
    """Sigmoid 函数：把任意数字压缩到 (0, 1)。"""
    return 1.0 / (1.0 + np.exp(-x))

def bradley_terry_prob(r_w, r_l):
    """
    Bradley-Terry 模型：计算「chosen (r_w) 比 rejected (r_l) 好」的概率。
    P(w > l) = sigmoid(r_w - r_l)
    """
    return sigmoid(r_w - r_l)

def rm_loss(r_w, r_l):
    """
    Reward Model 的损失函数（最大化对数似然的负值 = 最小化负对数似然）。
    L = -log(sigmoid(r_w - r_l))
    """
    return -np.log(bradley_terry_prob(r_w, r_l) + 1e-8)

# 演示：当 chosen 比 rejected 分高时，loss 应该很小
examples = [
    (2.0, -1.0, '明显更好'),
    (1.0, 0.5,  '稍微更好'),
    (0.5, 0.5,  '分不清楚'),
    (-0.5, 1.0, '反了！chosen 比 rejected 差'),
]
print(f'{'场景':<20} {'chosen分':>8} {'rejected分':>10} {'胜出概率':>8} {'Loss':>8}')
print('-' * 60)
for r_w, r_l, desc in examples:
    prob = bradley_terry_prob(r_w, r_l)
    loss = rm_loss(r_w, r_l)
    print(f'{desc:<20} {r_w:>8.2f} {r_l:>10.2f} {prob:>8.4f} {loss:>8.4f}')

In [ ]:
# 可视化：Bradley-Terry 概率曲线
delta = np.linspace(-4, 4, 200)  # r_w - r_l 的范围
prob = sigmoid(delta)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(delta, prob, 'b-', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='P=0.5 (无差异)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.fill_between(delta, 0.5, prob, where=(delta > 0), alpha=0.2, color='green', label='chosen 更好')
ax.fill_between(delta, prob, 0.5, where=(delta < 0), alpha=0.2, color='red', label='chosen 更差')
ax.set_xlabel('r_chosen - r_rejected（分数差）')
ax.set_ylabel('P(chosen 比 rejected 好)')
ax.set_title('Bradley-Terry 偏好模型：分数差 vs 概率')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/assets/bt_prob.png', dpi=100, bbox_inches='tight')
plt.show()
print('Bradley-Terry 概率曲线已保存')

In [ ]:
# =============================================================
# Part 2: 训练一个简化 Reward Model
# =============================================================

# 场景：用 5 个特征来表示一个回答的质量
# 特征：[相关性, 清晰度, 无害性, 简洁性, 准确性]（人工标注）
# 真实 RM 的输入是 token 序列，这里用特征向量模拟

def reward_model(features, weights):
    """简化的 Reward Model：特征加权求和。"""
    return float(np.dot(features, weights))

def train_reward_model(preference_data, n_features=5, n_epochs=200, lr=0.05):
    """
    用偏好对数据训练 Reward Model。
    preference_data: list of (chosen_features, rejected_features)
    """
    weights = np.zeros(n_features)
    losses = []

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        grad = np.zeros(n_features)

        for chosen, rejected in preference_data:
            r_w = reward_model(chosen, weights)
            r_l = reward_model(rejected, weights)
            prob = bradley_terry_prob(r_w, r_l)
            loss = -np.log(prob + 1e-8)
            epoch_loss += loss

            # 梯度：dL/dw = -(1 - prob) * (chosen - rejected)
            grad += -(1 - prob) * (chosen - rejected)

        weights -= lr * grad / len(preference_data)
        losses.append(epoch_loss / len(preference_data))

    return weights, losses

# 生成模拟偏好数据：好的回答各维度分数高，差的回答分数低
np.random.seed(0)
n_samples = 100
n_features = 5

# 好回答：特征均值在 0.7 左右；差回答：特征均值在 0.3 左右
chosen_features = np.random.beta(7, 3, (n_samples, n_features))   # 偏高
rejected_features = np.random.beta(3, 7, (n_samples, n_features)) # 偏低
preference_data = list(zip(chosen_features, rejected_features))

weights, losses = train_reward_model(preference_data)
print('训练完成！')
print(f'最终权重（各维度重要性）: {weights.round(3)}')
print(f'所有权重均为正数（高分特征 → 高奖励）: {all(w > 0 for w in weights)}')

In [ ]:
# 可视化训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss 下降曲线
ax1.plot(losses, 'b-', linewidth=1.5)
ax1.set_xlabel('训练轮次')
ax1.set_ylabel('平均 Loss')
ax1.set_title('Reward Model 训练曲线')
ax1.grid(alpha=0.3)

# 权重可视化
feature_names = ['相关性', '清晰度', '无害性', '简洁性', '准确性']
colors = ['#2196F3' if w > 0 else '#F44336' for w in weights]
ax2.barh(feature_names, weights, color=colors)
ax2.set_xlabel('权重（正值=高分特征更受偏好）')
ax2.set_title('Reward Model 学到的特征权重')
ax2.grid(alpha=0.3, axis='x')
ax2.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../docs/assets/rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print('Reward Model 训练图已保存')

In [ ]:
# 验证：RM 是否真的能分辨好坏？
# 对每个偏好对，检查 RM 是否给 chosen 打了更高的分
correct = 0
for chosen, rejected in preference_data:
    r_w = reward_model(chosen, weights)
    r_l = reward_model(rejected, weights)
    if r_w > r_l:
        correct += 1

accuracy = correct / len(preference_data)
print(f'偏好准确率: {accuracy:.1%}（{correct}/{len(preference_data)} 对排序正确）')
print('RM 已经学会了人类偏好：chosen 的分数通常高于 rejected')

In [ ]:
# =============================================================
# Part 3: KL 散度——PPO 的「刹车」
# =============================================================

def kl_divergence(p, q, eps=1e-10):
    """
    KL 散度：衡量分布 P 与 Q 的差异。
    KL(P||Q) = sum(P * log(P/Q))
    KL = 0 当且仅当 P = Q
    """
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    # 归一化
    p = p / p.sum()
    q = q / q.sum()
    return float(np.sum(p * np.log((p + eps) / (q + eps))))

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# 演示：SFT 模型 vs PPO 更新后的模型
vocab_size = 10
np.random.seed(1)

# SFT 模型的 token 概率分布（初始参考分布）
sft_logits = np.random.randn(vocab_size)
sft_probs = softmax(sft_logits)

# 三种 PPO 更新后的分布（小步、中步、大步）
scenarios = [
    ('小幅更新 (beta=0.1 约束)', softmax(sft_logits + 0.2 * np.random.randn(vocab_size))),
    ('中幅更新 (beta=0.01 约束)', softmax(sft_logits + 1.0 * np.random.randn(vocab_size))),
    ('大幅更新 (无 KL 约束)',  softmax(sft_logits + 5.0 * np.random.randn(vocab_size))),
]

print('KL 散度演示：PPO 更新后与 SFT 基线的距离')
print('-' * 55)
for name, ppo_probs in scenarios:
    kl = kl_divergence(ppo_probs, sft_probs)
    print(f'{name}: KL = {kl:.4f}')
print()
print('KL = 0 表示与原始 SFT 完全一样；KL 越大，偏离越远')

In [ ]:
# 可视化：三种更新幅度下的概率分布对比
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.arange(vocab_size)

for ax, (name, ppo_probs) in zip(axes, scenarios):
    kl = kl_divergence(ppo_probs, sft_probs)
    ax.bar(x - 0.2, sft_probs, width=0.4, label='SFT（基线）', alpha=0.8, color='blue')
    ax.bar(x + 0.2, ppo_probs, width=0.4, label='PPO 更新后', alpha=0.8, color='orange')
    ax.set_title(f'{name}\nKL={kl:.3f}', fontsize=9)
    ax.set_xlabel('Token')
    ax.set_ylabel('概率')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('KL 散度：PPO 更新幅度 vs 偏离 SFT 的程度', y=1.02)
plt.tight_layout()
plt.savefig('../docs/assets/kl_divergence.png', dpi=100, bbox_inches='tight')
plt.show()
print('KL 散度对比图已保存')

In [ ]:
# =============================================================
# Part 4: PPO 目标函数的简化模拟
# =============================================================

def ppo_objective(rm_score, kl, beta=0.05):
    """
    PPO 在 RLHF 中的目标（每次生成的奖励信号）。
    J = rm_score - beta * kl
    rm_score: Reward Model 打的分（越高越好）
    kl: 与 SFT 基线的 KL 散度（太大会被惩罚）
    beta: 平衡系数
    """
    return rm_score - beta * kl

# 模拟：不同 KL 下的 PPO 目标值
rm_scores = np.linspace(-1, 3, 100)  # RM 分数从低到高
kl_values = [0.5, 2.0, 5.0]          # 三种 KL 水平
beta = 0.1

fig, ax = plt.subplots(figsize=(8, 5))
for kl in kl_values:
    objectives = [ppo_objective(r, kl, beta) for r in rm_scores]
    ax.plot(rm_scores, objectives, linewidth=2, label=f'KL={kl:.1f} (惩罚={beta*kl:.2f})')

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Reward Model 分数')
ax.set_ylabel('PPO 目标值 (越高越好)')
ax.set_title(f'PPO 目标：RM分数 - beta*KL (beta={beta})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/assets/ppo_objective.png', dpi=100, bbox_inches='tight')
plt.show()
print('KL 惩罚越大，即使 RM 分数高，最终目标值也会被压低')
print('这就是为什么需要 beta 系数：防止模型走极端')

In [ ]:
# =============================================================
# Part 5: Reward Hacking 演示——为什么 KL 惩罚必不可少
# =============================================================

# 场景：语言模型在没有 KL 约束时，发现了一个钻空子的策略：
# 只输出「非常好！」这句话——RM 误打高分，但完全没用

class MockLanguageModel:
    def __init__(self, strategy='normal'):
        self.strategy = strategy

    def generate(self, prompt):
        if self.strategy == 'hacking':
            # Reward Hacking：找到让 RM 打高分的固定模板
            return '非常感谢您的提问！这个问题非常重要，我完全同意您的观点。总之，一切都很好！'
        else:
            return f'[正常回答 {prompt} 的内容...]'

class MockRewardModel:
    def score(self, response):
        # 简化的 RM：偏好「非常」、「感谢」、「重要」这类词
        keywords = ['非常', '感谢', '重要', '同意', '好']
        score = sum(response.count(kw) for kw in keywords) * 0.5
        return score

rm = MockRewardModel()
normal_model = MockLanguageModel('normal')
hacking_model = MockLanguageModel('hacking')

prompt = '请解释什么是黑洞'
normal_resp = normal_model.generate(prompt)
hacking_resp = hacking_model.generate(prompt)

normal_score = rm.score(normal_resp)
hacking_score = rm.score(hacking_resp)

print('=== Reward Hacking 演示 ===\n')
print(f'正常回答: "{normal_resp}"')
print(f'RM 分数: {normal_score:.1f}\n')
print(f'Hacking 回答: "{hacking_resp}"')
print(f'RM 分数: {hacking_score:.1f}\n')
print('---')
print(f'Hacking 分数比正常高 {hacking_score - normal_score:.1f} 分，但完全没用！')
print('这就是为什么需要 KL 惩罚：让模型不能走太偏')

In [ ]:
# =============================================================
# Part 6: RLHF 全流程串联
# =============================================================

print('=== RLHF 全流程模拟 ===\n')

# ── 阶段1：SFT ──
print('【阶段1】SFT（监督微调）')
print('  人工标注员为 1000 个 prompt 写出理想回答')
print('  用这些示范数据微调 GPT-3 → 得到 SFT 模型')
print(f'  示范数据量: ~13000 条（来自 OpenAI 论文）')
print()

# ── 阶段2：Reward Model ──
print('【阶段2】Reward Model 训练')
print('  对每个 prompt，让 SFT 生成 4-9 个候选回答')
print('  标注员对候选回答排序（比写示范快 4x）')
print('  训练 RM 学会预测人类偏好')
print(f'  偏好对数量: ~33000 对（来自 OpenAI 论文）')
print()

# ── 阶段3：PPO ──
print('【阶段3】PPO 强化学习')
print('  对新 prompt，SFT 模型生成回答')
print('  RM 给回答打分 → 作为强化学习奖励')
print('  PPO 更新模型参数（附 KL 惩罚防止走极端）')
print('  反复迭代 → InstructGPT')
print()

# ── 结果 ──
print('=== 关键结果 ===')
print('  1.3B InstructGPT vs 175B GPT-3：')
print('  → 人类评估者 85% 的情况下更偏好 InstructGPT')
print('  → 参数量相差 134 倍，但对齐的小模型胜过未对齐的大模型！')
print()
print('  2022年11月30日：ChatGPT 上线')
print('  → 5天用户突破100万，2个月月活超过1亿')
print('  → 有史以来增长最快的消费者应用')

In [ ]:
# InstructGPT 模型系列对比
fig, ax = plt.subplots(figsize=(10, 5))

models = ['GPT-3\n175B\n(未对齐)', 'InstructGPT\n1.3B\n(RLHF)', 'InstructGPT\n6B\n(RLHF)', 'InstructGPT\n175B\n(RLHF)']
# 人类偏好评分（论文中的相对评分，以 GPT-3 175B 为基准 50%）
human_preference = [15, 85, 87, 88]  # 与 GPT-3 175B 的对比胜率
params = [175, 1.3, 6, 175]  # 参数量 B

colors = ['#F44336', '#4CAF50', '#2196F3', '#9C27B0']
bars = ax.bar(models, human_preference, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(50, color='gray', linestyle='--', linewidth=1.5, label='基准线 (50% = 平手)')

for bar, pref in zip(bars, human_preference):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{pref}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('人类偏好率（vs GPT-3 175B）')
ax.set_title('InstructGPT vs GPT-3：对齐让小模型胜过大模型\n（Ouyang et al. 2022, arXiv:2203.02155）')
ax.set_ylim(0, 100)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../docs/assets/instructgpt_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('InstructGPT 模型对比图已保存')
print('\n关键洞察：1.3B 的对齐模型胜过 175B 的未对齐模型！')
print('对齐（Alignment）与规模（Scale）同样重要。')

In [ ]:
# =============================================================
# 总结：RLHF 教会了我们什么
# =============================================================

summary = {
    'RLHF 三阶段': ['SFT（有示范数据做监督微调）',
                  'Reward Model（用偏好排序训练打分器）',
                  'PPO（用 RM 分数做强化学习 + KL 约束）'],
    '核心数学': ['Bradley-Terry: P(w>l) = σ(r_w - r_l)',
              'RM Loss: -log(σ(r_w - r_l))',
              'PPO Goal: r_RM(x,y) - β·KL(π||π_SFT)'],
    '关键洞察': ['1.3B InstructGPT > 175B GPT-3（对齐让小模型胜大模型）',
              'Reward Hacking 是真实威胁（KL 惩罚必不可少）',
              'RLHF 开启了现代 LLM 对齐的标准范式'],
    '局限': ['RM 只能学到标注员的偏好（可能有偏差）',
           'PPO 训练不稳定（后来 DPO 尝试绕开它）',
           '仍有幻觉，不能彻底消除'],
}

for section, items in summary.items():
    print(f'\n【{section}】')
    for item in items:
        print(f'  • {item}')